In [1]:
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("INSTALL spatial; LOAD spatial;")
con.sql("SET s3_region='us-west-2';")

PLACES = "s3://overturemaps-us-west-2/release/2026-09-23.1/theme=places/type=place/*.parquet"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [2]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{PLACES}')")

┌──────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │                                                                                                                         column_type                                                                                                                         │  null   │   key   │ default │  extra  │
│     varchar      │                                                                                                                           varchar                                                                                                                           │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼───────────────────────────────────

In [3]:
con.sql(f"""
    SELECT basic_category, COUNT(*) AS places
    FROM read_parquet('{PLACES}')
    WHERE bbox.xmin BETWEEN 34.5 AND 55.7
      AND bbox.ymin BETWEEN 16.3 AND 32.2
      AND (basic_category ILIKE '%pharm%'
        OR basic_category ILIKE '%gas%'
        OR basic_category ILIKE '%fuel%'
        OR basic_category ILIKE '%cafe%'
        OR basic_category ILIKE '%coffee%'
        OR basic_category ILIKE '%restaurant%'
        OR basic_category ILIKE '%bank%'
        OR basic_category ILIKE '%atm%')
    GROUP BY basic_category
    ORDER BY places DESC
""").show(max_rows=60)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────┬────────┐
│        basic_category        │ places │
│           varchar            │ int64  │
├──────────────────────────────┼────────┤
│ restaurant                   │  48233 │
│ coffee_shop                  │  12102 │
│ cafe                         │   8641 │
│ fast_food_restaurant         │   6520 │
│ pharmacy_and_drug_store      │   5570 │
│ bank_or_credit_union         │   4330 │
│ gas_station                  │   3291 │
│ atm                          │    627 │
│ fueling_station              │    158 │
│ natural_gas_utility_provider │      1 │
│ food_bank                    │      1 │
└──────────────────────────────┴────────┘
  11 rows                     2 columns



In [4]:
services = con.sql(f"""
    SELECT
        id,
        names.primary AS name,
        CASE
            WHEN basic_category IN ('restaurant', 'fast_food_restaurant') THEN 'restaurant'
            WHEN basic_category IN ('coffee_shop', 'cafe') THEN 'cafe'
            WHEN basic_category = 'pharmacy_and_drug_store' THEN 'pharmacy'
            WHEN basic_category IN ('bank_or_credit_union', 'atm') THEN 'bank'
            WHEN basic_category IN ('gas_station', 'fueling_station') THEN 'fuel'
        END AS service_type,
        basic_category,
        addresses[1].locality AS city,
        ST_Y(geometry) AS latitude,
        ST_X(geometry) AS longitude,
        confidence,
        operating_status
    FROM read_parquet('{PLACES}')
    WHERE bbox.xmin BETWEEN 34.5 AND 55.7
      AND bbox.ymin BETWEEN 16.3 AND 32.2
      AND addresses[1].country = 'SA'
      AND basic_category IN ('restaurant', 'fast_food_restaurant', 'coffee_shop', 'cafe',
                             'pharmacy_and_drug_store', 'bank_or_credit_union', 'atm',
                             'gas_station', 'fueling_station')
""").df()

services.to_csv("data/raw/services_overture.csv", index=False)
services.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(32566, 9)